# ONPE ERM2022 – Scraping Actas por Ubigeo
Notebook para extraer **organización política → votos (y porcentaje)** por **Ubigeo** desde:
`https://resultadoshistorico.onpe.gob.pe/ERM2022/EleccionesMunicipales/RePro`.


In [ ]:
pip install selenium webdriver-manager requests beautifulsoup4 pandas

Note: you may need to restart the kernel to use updated packages.


In [ ]:

import time
import requests
import pandas as pd
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager



In [ ]:
WIKI_URLS = [
    "https://es.wikipedia.org/wiki/Anexo:Distritos_del_Per%C3%BA",
    "https://it.wikipedia.org/wiki/Distretti_del_Per%C3%B9",
]
BASE_ONPE_URL = (
    "https://resultadoshistorico.onpe.gob.pe/ERM2022/EleccionesMunicipales/"
    "RePro/{dep}/{prov}/{dist}"
)
OUTPUT_CSV = "erm2022_por_distrito_ubigeo_selenium.csv"
PAUSA_ENTRE_DISTRITOS = 0.7  # segundos


In [ ]:
def obtener_ubigeos_desde_wikipedia():
    """
    Descarga la lista de distritos y ubigeos desde Wikipedia
    (ES o IT), respetando la política de User-Agent.

    Devuelve un DataFrame con columnas:
    - ubigeo (6 dígitos)
    - distrito
    - provincia (si está disponible en la tabla)
    - region   (si está disponible en la estructura de encabezados)
    """
    print("Descargando lista de distritos y ubigeos desde Wikipedia...")

    headers = {
        # pon aquí algo identificable (mejor si incluyes tu correo institucional)
        "User-Agent": "KarlaUbigeoScraper/0.1 (contacto@ejemplo.com)"
    }

    resp = None
    last_exc = None

    # Intentar con varias URLs de Wikipedia
    for url in WIKI_URLS:
        try:
            print(f"  Probando {url}")
            resp = requests.get(url, headers=headers, timeout=30)
            resp.raise_for_status()
            print(f"  OK -> {url}")
            break
        except Exception as e:
            print(f"  Falló {url}: {e}")
            last_exc = e
            resp = None

    if resp is None:
        # si todas fallan, relanzamos el último error
        raise last_exc

    soup = BeautifulSoup(resp.text, "html.parser")
    content = soup.find("div", id="bodyContent") or soup

    region_actual = None
    provincia_actual = None
    regs = []

    # Recorremos elementos del contenido en orden
    for el in content.descendants:
        if not hasattr(el, "name"):
            continue

        # Posibles encabezados de región / provincia
        if el.name in ("h2", "h3", "h4"):
            texto = el.get_text(strip=True)

            # muy genérico: busca “Región” / “Regione” y “Provincia”
            if "Región" in texto or "Regione" in texto:
                region_actual = (
                    texto.replace("Región de", "")
                         .replace("Regione di", "")
                         .replace("Región", "")
                         .strip()
                )
            elif "Provincia" in texto:
                provincia_actual = (
                    texto.replace("Provincia de", "")
                         .replace("Provincia di", "")
                         .replace("Provincia", "")
                         .strip()
                )

        # Tablas que tengan columna UBIGEO
        elif el.name == "table":
            ths = el.find_all("th")
            if not ths:
                continue

            headers_txt = [th.get_text(strip=True).upper() for th in ths]
            if "UBIGEO" not in "".join(headers_txt):
                continue

            # Identificamos índice de columna UBIGEO y distrito
            idx_ubi = None
            idx_dist = None
            for i, h in enumerate(headers_txt):
                if "UBIGEO" in h:
                    idx_ubi = i
                # columna distrito puede llamarse "Distrito", "Distretto", etc.
                if any(w in h for w in ["DISTRITO", "DISTRETTO"]):
                    idx_dist = i

            # fallback simple: asumimos [UBIGEO, Distrito, ...]
            if idx_ubi is None:
                idx_ubi = 0
            if idx_dist is None:
                idx_dist = 1

            # procesar filas
            for fila in el.find_all("tr")[1:]:
                celdas = fila.find_all("td")
                if len(celdas) <= max(idx_ubi, idx_dist):
                    continue

                ub_txt = celdas[idx_ubi].get_text(strip=True)
                dist_txt = celdas[idx_dist].get_text(strip=True)

                ub_dig = "".join(ch for ch in ub_txt if ch.isdigit())
                if len(ub_dig) != 6:
                    continue

                regs.append({
                    "ubigeo": ub_dig,
                    "distrito": dist_txt,
                    "provincia": provincia_actual,
                    "region": region_actual,
                })

    df = pd.DataFrame(regs).drop_duplicates(subset=["ubigeo"])
    print(f"Se obtuvieron {len(df)} ubigeos distintos.")
    return df


In [ ]:
def crear_driver(headless=True):
    chrome_options = Options()
    if headless:
        chrome_options.add_argument("--headless=new")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")

    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=chrome_options)
    return driver


def limpiar_numero(texto: str):
    if not texto:
        return None
    t = (
        texto.replace(",", "")
             .replace(".", "")
             .replace(" ", "")
    )
    return int(t) if t.isdigit() else None


def extraer_tabla_partidos(driver):
    """
    Busca la tabla con cabecera:
    Organización política | Total | % Votos válidos | % Votos emitidos
    y devuelve lista de tuplas (organizacion_politica, total_votos).
    """
    wait = WebDriverWait(driver, 20)

    tabla = wait.until(
        EC.presence_of_element_located(
            (
                By.XPATH,
                "//table[.//th[contains(translate(., "
                "'óÓáÁéÉíÍúÚñÑ', 'oOaAeEiIuUnN'),"
                "'ORGANIZACION POLITICA')]]"
            )
        )
    )

    driver.execute_script(
        "arguments[0].scrollIntoView({block: 'center'});", tabla
    )

    filas = tabla.find_elements(By.XPATH, ".//tbody/tr")
    resultados = []

    for tr in filas:
        celdas = tr.find_elements(By.TAG_NAME, "td")
        if len(celdas) < 2:
            continue

        org = celdas[0].text.strip()
        total_txt = celdas[1].text.strip()
        if not org:
            continue

        up = org.upper()
        # si quieres incluir blancos/nulos/totales, comenta este bloque
        if any(x in up for x in [
            "TOTAL DE VOTOS VÁLIDOS",
            "TOTAL DE VOTOS EMITIDOS",
            "VOTOS EN BLANCO",
            "VOTOS NULOS",
        ]):
            continue

        total = limpiar_numero(total_txt)
        if total is None:
            continue

        resultados.append((org, total))

    return resultados


def obtener_resultados_distrito(driver, ubigeo):
    """
    Construye la URL de ONPE para ese ubigeo usando el patrón:
    dep = XX0000
    prov = XXXX00
    dist = XXXXXX (ubigeo)
    """
    dep = ubigeo[:2] + "0000"
    prov = ubigeo[:4] + "00"
    dist = ubigeo

    url = BASE_ONPE_URL.format(dep=dep, prov=prov, dist=dist)
    print(f"  > {ubigeo} -> {url}")

    try:
        driver.get(url)
    except Exception as e:
        print(f"    [ERROR al abrir URL] {e}")
        return []

    try:
        filas = extraer_tabla_partidos(driver)
    except Exception as e:
        print(f"    [ERROR al leer tabla] {e}")
        return []

    resultados = []
    for org, total in filas:
        resultados.append({
            "ubigeo": ubigeo,
            "organizacion_politica": org,
            "total_votos": total,
            "url_origen": url,
        })
    return resultados


In [16]:
def main():
    # 1) Sacar todos los ubigeos y distritos desde Wikipedia
    df_ubi = obtener_ubigeos_desde_wikipedia()

    # Si quieres filtrar para probar, descomenta esta línea:
    # df_ubi = df_ubi[df_ubi["provincia"] == "Lima"].head(5)

    driver = crear_driver(headless=True)
    registros = []

    try:
        for _, row in df_ubi.iterrows():
            ubigeo = row["ubigeo"]
            distrito = row["distrito"]
            provincia = row["provincia"]
            region = row["region"]

            print(f"\nUbigeo {ubigeo} - {region} / {provincia} / {distrito}")

            resultados = obtener_resultados_distrito(driver, ubigeo)

            for r in resultados:
                registros.append({
                    "ubigeo": ubigeo,
                    "region": region,
                    "provincia": provincia,
                    "distrito": distrito,
                    "organizacion_politica": r["organizacion_politica"],
                    "total_votos": r["total_votos"],
                    "url_origen": r["url_origen"],
                })

            time.sleep(PAUSA_ENTRE_DISTRITOS)

    finally:
        driver.quit()

    df_final = pd.DataFrame(registros)
    df_final.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    print(f"\nListo. Datos guardados en: {OUTPUT_CSV}")


if __name__ == "__main__":
    main()

Descargando lista de distritos y ubigeos desde Wikipedia...
  Probando https://es.wikipedia.org/wiki/Anexo:Distritos_del_Per%C3%BA
  Falló https://es.wikipedia.org/wiki/Anexo:Distritos_del_Per%C3%BA: 404 Client Error: Not Found for url: https://es.wikipedia.org/wiki/Anexo:Distritos_del_Per%C3%BA
  Probando https://it.wikipedia.org/wiki/Distretti_del_Per%C3%B9
  OK -> https://it.wikipedia.org/wiki/Distretti_del_Per%C3%B9
Se obtuvieron 1872 ubigeos distintos.

Ubigeo 010101 - Amazonas / Chachapoyas / Chachapoyas
  > 010101 -> https://resultadoshistorico.onpe.gob.pe/ERM2022/EleccionesMunicipales/RePro/010000/010100/010101
    [ERROR al leer tabla] Message: 
Stacktrace:
0   chromedriver                        0x00000001009b3aa4 cxxbridge1$str$ptr + 2943380
1   chromedriver                        0x00000001009ab760 cxxbridge1$str$ptr + 2909776
2   chromedriver                        0x00000001004c22bc _RNvCsgXDX2mvAJAg_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 74028
3   chromedriver

KeyboardInterrupt: 